<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/35_hybrid_routing_rag/hybrid_routing_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy pandas

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
space_docs = [
    "Elon Musk founded SpaceX.",
    "SpaceX works on rockets and space exploration."
]

auto_docs = [
    "Tesla builds electric cars.",
    "Elon Musk is CEO of Tesla."
]

space_df = pd.DataFrame({"text": space_docs})
auto_df = pd.DataFrame({"text": auto_docs})

In [ ]:
def classify_query(query):
    q = query.lower()

    if any(word in q for word in ["rocket", "space", "spacex"]):
        return "space"
    elif any(word in q for word in ["car", "tesla", "electric"]):
        return "automobile"
    else:
        return "unknown"

In [ ]:
def retrieve(query, df):
    query_words = query.lower().split()
    scores = []

    for text in df["text"]:
        text_lower = text.lower()

        # basic match
        score = sum(word in text_lower for word in query_words)

        # 🔥 EXTRA BOOST for semantic hint
        if "tesla" in query.lower() and "tesla" in text_lower:
            score += 1
        if "spacex" in query.lower() and "spacex" in text_lower:
            score += 1

        scores.append(score)

    scores = np.array(scores)
    top_idx = np.argmax(scores)

    return df.iloc[top_idx]["text"], scores[top_idx]

In [ ]:
def extract_answer(context):
    if "founded" in context.lower():
        return "Elon Musk"
    elif "spacex" in context.lower():
        return "SpaceX"
    elif "tesla" in context.lower():
        return "Tesla"
    else:
        return "Answer not found"

In [ ]:
def hybrid_rag(query):
    print("\n==============================")

    domain = classify_query(query)

    if domain == "unknown":
        print("Query:", query)
        print("Rejected 🚫")
        print("Reason: Unknown domain")
        return

    # Retrieve
    if domain == "space":
        context, score = retrieve(query, space_df)
    else:
        context, score = retrieve(query, auto_df)

    # Confidence check
    if score <= 0:
        print("Query:", query)
        print("Fallback 🚫")
        print("Reason: Low confidence retrieval")
        return

    # Answer
    answer = extract_answer(context)

    print("Query:", query)
    print("Domain:", domain)
    print("Score:", score)
    print("Context:", context)
    print("Answer:", answer)

In [19]:
hybrid_rag("Which company works on rockets?")
hybrid_rag("Who founded Tesla?")
hybrid_rag("What is weather today?")
hybrid_rag("Tell me about electric rockets")


Query: Which company works on rockets?
Domain: space
Score: 2
Context: SpaceX works on rockets and space exploration.
Answer: SpaceX

Query: Who founded Tesla?
Domain: automobile
Score: 1
Context: Tesla builds electric cars.
Answer: Tesla

Query: What is weather today?
Rejected 🚫
Reason: Unknown domain

Query: Tell me about electric rockets
Domain: space
Score: 1
Context: SpaceX works on rockets and space exploration.
Answer: SpaceX
